# LaTeX로 풀이 단계 작성하기

문제 1·2·3은 각 단계의 LaTeX만 작성합니다. `flow.play_latex(...)`가 식의 파싱·매핑·배치·재생을 처리합니다. 수식은 `r"..."` 형식으로 입력하세요.

## 문제 이미지와 카메라

`flow.load_problem()`은 `SolutionFlow`에 지정한 PDF에서 현재 문제 이미지를 읽어 화면 상단에 배치합니다. 따라서 PDF를 사용할 때는 장면을 `ProblemScene`으로 만들고, `SolutionFlow(self, pdf_path, problem_number)`를 만든 뒤 호출해야 합니다.

```python
class Problem1(ProblemScene):
    def construct(self):
        flow = SolutionFlow(self, "math_2709.pdf", 1)
        flow.load_problem()
```

PDF 없이 수식만 확인하려면 `SolutionFlow(self, None, 번호)`로 만들 수 있습니다. 이 경우 `load_problem()`은 아무것도 추가하지 않습니다.

`ProblemScene`은 문제 영역과 풀이 영역의 위치·너비를 준비합니다. `play_latex()`, `begin()`, `transition()`은 기본적으로 현재 표시할 식에 카메라를 부드럽게 포커싱합니다. 직접 카메라를 이동하려면 다음처럼 `flow.focus(eq)`를 사용합니다.

```python
eqs = flow.latex_steps(r"x+1", r"=2")
flow.stack(*eqs)
flow.begin(eqs[0])
flow.focus(eqs[1])
flow.transition(eqs[0], eqs[1], focus=False)
```

카메라 동작은 `SolutionFlow` 생성 때 조정할 수 있습니다. `camera_width`는 포커스 화면의 최소 너비이고, `camera_run_time`은 카메라 이동 시간입니다. 개별 전환에서 `focus=False`를 지정하면 해당 전환에서는 카메라를 움직이지 않습니다.

문제 2에서는 다항식 미분과 미분계수의 극한 정의를 연결하고, 앞의 도함수로 돌아가 `f′(2)`의 2를 대입합니다. 문제 3에서는 `a_2` 전체가 2로 바뀌고, 앞서 구한 `d=3`에서 3을 가져옵니다. 임의의 모든 수학 변형을 자동 증명하는 기능은 아닙니다.

`True/False`를 각 식 뒤에 붙이면 다음 전환의 같은 줄 여부를 지정할 수 있습니다. 예를 들어 `r"=2", True`는 직전 식과 같은 자리에서 교체하고, `False`는 다음 줄에 표시합니다. 기존처럼 `same_line=[...]` 목록을 keyword 인자로 전달하는 방식도 사용할 수 있습니다.

원본 변수는 보존됩니다. `strict=True`는 휴리스틱 값 변경을 거부합니다.

먼저 정의하려면 `eq1, eq1_1, ... = flow.latex_steps(...)` 후 `flow.stack(...)`, `flow.play_linear()`를 호출하세요. `sources`로 출발식 번호(0부터)를 지정하거나 `links`로 토큰 연결을 보정할 수 있습니다. 자세한 설명은 README.md를 참고하세요.


In [2]:
%load_ext autoreload
%autoreload 2

import os
import warnings

warnings.filterwarnings(
    "ignore",
    message="This method is not guaranteed to stay around.*",
    category=DeprecationWarning,
)

miktex_bin = r"C:\Users\hbkim\AppData\Local\Programs\MiKTeX\miktex\bin\x64"
if miktex_bin not in os.environ["PATH"]:
    os.environ["PATH"] = miktex_bin + os.pathsep + os.environ["PATH"]

from manim import *
from reactive_manim import *
from manim.utils.ipython_magic import ManimMagic

from module.mobjects import *
from module.simple_solution_flow import SolutionFlow

get_ipython().register_magics(ManimMagic)

# 원본 문제 PDF 없이 수식과 애니메이션만 검증할 때 False
USE_PDF = True


In [3]:
%%manim -ql -v WARNING Problem1

class Problem1(ProblemScene):
    def construct(self):
        flow = SolutionFlow(self, "math_2709.pdf" if USE_PDF else None, 1)
        flow.load_problem()

        # 이전 단계의 항을 참조할 필요 없이 LaTeX만 작성한다.
        flow.play_latex(
            r"2^{\frac{3}{2}} \times 4^{-\frac{1}{2}}",
            r"= 2^{\frac{3}{2}} \times (2^2)^{-\frac{1}{2}}",
            r"= 2^{\frac{3}{2}} \times 2^{-1}",
            r"= 2^{\frac{3}{2}-1}",
            r"= 2^{\frac{1}{2}}",
        )
        flow.finish()


Manim Community v0.21.0

In [4]:
%%manim -ql -v WARNING Problem2

class Problem2(ProblemScene):
    def construct(self):
        flow = SolutionFlow(self, "math_2709.pdf" if USE_PDF else None, 2)
        flow.load_problem()

        flow.play_latex(
            r"f(x)=2x^3-x-4",
            r"\Rightarrow f^{\prime}(x)=6x^2-1",
            r"\lim_{x\to2}\frac{f(x)-f(2)}{x-2}",
            r"=f^{\prime}(2)",
            r"=6x^2-1",
            r"=6\cdot2^2-1",
            r"=23",
            same_line=[False, False, False, False, True, True],
            strict=True,
        )
        flow.finish()


Manim Community v0.21.0

In [5]:
%%manim -ql -v WARNING Problem3

class Problem3(ProblemScene):
    def construct(self):
        flow = SolutionFlow(self, "math_2709.pdf" if USE_PDF else None, 3)
        flow.load_problem()

        flow.play_latex(
            r"a_{10}-a_7=9",
            r"a_{10}-a_7=3d=9", True,
            r"d=\frac{9}{3}", False,
            r"d=3", True,
            r"a_7", False,
            r"a_7=a_2+5d", True,
            r"a_7=2+5\cdot3", False,
            r"a_7=17", True,
            strict=True,
        )
        flow.finish()


Manim Community v0.21.0

In [14]:
%%manim -ql -v WARNING Problem4

class Problem4(ProblemScene):
    def construct(self):
        flow = SolutionFlow(self, "math_2709.pdf" if USE_PDF else None, 4)
        flow.load_problem()

        # x=2에서 좌우식의 함수값이 같아야 연속이다.
        flow.play_latex(
            r"\lim_{x\to2-}f(x)",
            r"\lim_{x\to2-}f(x)=7x+a\mid_{x=2}", True,
            r"\lim_{x\to2-}f(x)=7\cdot2+a", True,
            r"\lim_{x\to2-}f(x)=a+14", True,
            r"=\lim_{x\to2+}f(x)", False,
            r"=\lim_{x\to2+}f(x)=x^2+ax\mid_{x=2}", True,
            r"=\lim_{x\to2+}f(x)=2^2+2a", True,
            r"=\lim_{x\to2+}f(x)=2a+4", True,
            r"a+14=2a+4", False,
            r"10=a", False,
            r"a=10", True
            
            
            
            
            
        )
        flow.finish()


Manim Community v0.21.0